# Сравнение моделей визуального внимания в Google Colab
**Автор эксперимента: Меркурьев Захар Алексеевич**

Блокнот сравнивает прогноз **следующей реальной фиксации** у трёх методов:
1. **DeepGaze IIE** — статическое распределение внимания.
2. **DeepGaze IIE + IOR** — та же карта с простым подавлением посещённых областей.
3. **DeepGaze III** — последовательная модель, использующая четыре предыдущие фиксации.

Используются готовые веса и реальные данные свободного просмотра **OSIE**. Обучения нет.
Результат: таблицы CSV, графики PNG, прогнозы на примерах, описание протокола и ZIP-архив.

## Как запустить
1. Открой [Google Colab](https://colab.research.google.com/) → **Файл → Загрузить блокнот**.
2. **Среда выполнения → Сменить среду выполнения → GPU** (например, T4).
3. Запускай ячейки сверху вниз кнопкой ▶. Можно выбрать **Выполнить все**.
4. В последней ячейке скачается архив с результатами.

Первый запуск скачает библиотеки, веса моделей и выбранные изображения. Бесплатный GPU
предоставляется по доступности; точное время выполнения заранее неизвестно.
По умолчанию используется небольшая пробная выборка. **Не считай её итоговым исследованием.**

Блокнот подготовлен для Colab; полный запуск с весами моделей при подготовке не выполнялся.


## 1. Настройки
Для первого запуска оставь значения по умолчанию. Для основного эксперимента заранее выбери,
например, 100 изображений и больше наблюдателей, а затем не меняй выборку по результатам.
Это пример масштаба, а не расчёт необходимой статистической мощности.

Используется исходное разрешение OSIE, без изменения координат и размера изображения.
Каждый прогноз получает **только предыдущие реальные фиксации**, но не прогнозируемую точку.
Первые четыре фиксации служат контекстом и в метрики не входят.


In [ ]:
from pathlib import Path
import json, hashlib, random, sys, subprocess, os, time

SEED = 42
N_IMAGES = 10                 # пробный запуск; основной, например, 100
SUBJECTS_PER_IMAGE = 2        # от 1 до 15; для основного можно увеличить
MAX_TARGETS_PER_SUBJECT = 3   # прогнозы после первых 4 фиксаций; 0 = все доступные
IOR_STRENGTH = 2.0            # заранее фиксированный параметр, не подбирать на тесте
IOR_SIGMA_FRACTION = 0.05     # sigma = эта доля меньшей стороны изображения
N_BOOTSTRAP = 2000
N_EXAMPLES = 3
DEEPGAZE_REF = "main"         # для точного повторения вставь SHA из manifest.json

BASE = Path('/content/attention_experiment')
BASE.mkdir(parents=True, exist_ok=True)
CACHE = BASE / 'cache'
CACHE.mkdir(exist_ok=True)
assert 1 <= N_IMAGES <= 700
assert 1 <= SUBJECTS_PER_IMAGE <= 15
assert MAX_TARGETS_PER_SUBJECT >= 0 and IOR_STRENGTH >= 0
assert IOR_SIGMA_FRACTION > 0 and N_BOOTSTRAP >= 100
print('Выбрано изображений:', N_IMAGES)
print('Все вычисления и файлы будут созданы в среде Colab.')


## 2. Установка в Colab
Установка выполняется на сервере Colab. Исходный код берётся из репозитория авторов.
Фактическая версия кода и версии библиотек сохраняются в результатах.
Если Colab после установки попросит перезапуск среды, перезапусти её и снова выполняй ячейки сверху вниз.


In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'scipy', 'pandas', 'matplotlib', 'pillow', 'tqdm', 'requests'])
repo = BASE / 'DeepGaze'
if not (repo / '.git').exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/matthias-k/DeepGaze.git', str(repo)])
subprocess.check_call(['git', '-C', str(repo), 'fetch', 'origin', DEEPGAZE_REF])
subprocess.check_call(['git', '-C', str(repo), 'checkout', '--detach', 'FETCH_HEAD'])
DEEPGAZE_SHA = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', str(repo)])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.io import loadmat
from scipy.ndimage import zoom
from scipy.special import logsumexp
from tqdm.auto import tqdm
import requests
import torch
import deepgaze_pytorch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', DEVICE, '| версия кода:', DEEPGAZE_SHA)
if DEVICE == 'cpu':
    print('GPU не подключён: работать будет медленнее. При желании переключи среду на GPU.')


## 3. Загрузка небольшой части OSIE
Скачивается файл реальных фиксаций и только выбранные изображения из репозитория OSIE.
Изображения выбираются случайно с фиксированным seed **до получения прогнозов**.
Координаты читаются так же, как в загрузчике OSIE библиотеки pysaliency;
невалидные точки не обрезаются до границ картинки — соответствующие окна пропускаются и учитываются в отчёте.


In [ ]:
def download(url, path):
    path = Path(path)
    if path.exists() and path.stat().st_size:
        return path
    tmp = path.with_suffix(path.suffix + '.part')
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=(30, 180)) as response:
                response.raise_for_status()
                with tmp.open('wb') as f:
                    for chunk in response.iter_content(1024 * 1024):
                        f.write(chunk)
            tmp.replace(path)
            return path
        except Exception:
            if attempt == 2:
                raise
            time.sleep(2 * (attempt + 1))

OSIE_BASE = 'https://raw.githubusercontent.com/NUS-VIP/predicting-human-gaze-beyond-pixels/master/data/'
mat_path = download(OSIE_BASE + 'eye/fixations.mat', CACHE / 'fixations.mat')
mat_md5 = hashlib.md5(mat_path.read_bytes()).hexdigest()
assert mat_md5 == '8efdf6fe66f38b6e70f854c7ff45aa70', 'Файл OSIE изменился: проверь источник перед продолжением.'
raw = loadmat(mat_path)['fixations'].flatten()
records = {}
for record in raw:
    stimulus = record[0, 0]
    name = str(stimulus['img'][0])
    paths = []
    for subject, entry in enumerate(stimulus['subjects'].flatten()):
        fixation = entry[0, 0]
        xs = np.asarray(fixation['fix_x']).flatten().astype(float)
        ys = np.asarray(fixation['fix_y']).flatten().astype(float)
        assert len(xs) == len(ys)
        paths.append((subject, np.column_stack([xs, ys])))
    records[name] = paths

rng = np.random.default_rng(SEED)
names = sorted(records)
selected_names = sorted(rng.choice(names, size=N_IMAGES, replace=False).tolist())
images = {}
image_hashes = {}
for name in tqdm(selected_names, desc='Изображения OSIE'):
    path = download(OSIE_BASE + 'stimuli/' + name, CACHE / name)
    images[name] = np.asarray(Image.open(path).convert('RGB'))
    image_hashes[name] = hashlib.sha256(path.read_bytes()).hexdigest()

config = dict(seed=SEED, n_images=N_IMAGES, subjects_per_image=SUBJECTS_PER_IMAGE,
              max_targets_per_subject=MAX_TARGETS_PER_SUBJECT,
              ior_strength=IOR_STRENGTH, ior_sigma_fraction=IOR_SIGMA_FRACTION,
              bootstrap=N_BOOTSTRAP, examples=N_EXAMPLES, deepgaze_sha=DEEPGAZE_SHA,
              selected_images=selected_names, image_sha256=image_hashes,
              fixations_md5=mat_md5, protocol_version=1)
run_id = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]
OUT = BASE / ('results_' + run_id)
OUT.mkdir(exist_ok=True)
(OUT / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2))
print('Папка результатов:', OUT)
print('Изображения:', ', '.join(selected_names))


## 4. Протокол и метрики
**Основная метрика:** средний логарифм вероятности реальной следующей фиксации в битах,
относительно равномерного распределения: `log2(p(x,y) × W × H)`. Больше — лучше.
Это выигрыш относительно **равномерной модели**, а не информационный выигрыш относительно center bias.

**Дополнительная метрика:** NSS — значение стандартизированной карты вероятностей в реальной точке.
Больше — лучше. Обе метрики оценивают следующий шаг при известной реальной истории,
**не качество автономно сгенерированной траектории целиком**. ScanMatch/MultiMatch здесь не вычисляются.

Вариант IOR вычитает из логарифма карты сумму гауссовых штрафов вокруг последних четырёх фиксаций,
затем нормирует карту. Это простой авторский базовый метод, не обучение новой нейросети.
У IIE и III разные архитектуры: разницу качества нельзя приписать только наличию памяти.


In [ ]:
def normalize_log_density(values):
    values = np.asarray(values, dtype=np.float64).squeeze()
    if values.ndim != 2 or not np.isfinite(values).all():
        raise ValueError('Ожидалась конечная двумерная карта log probability.')
    return values - logsumexp(values)

def with_ior(logp, history):
    h, w = logp.shape
    yy, xx = np.mgrid[:h, :w]
    sigma = IOR_SIGMA_FRACTION * min(h, w)
    penalty = np.zeros((h, w), dtype=np.float64)
    for x, y in np.asarray(history)[-4:]:
        penalty += np.exp(-((xx - x)**2 + (yy - y)**2) / (2 * sigma**2))
    return normalize_log_density(logp - IOR_STRENGTH * penalty)

def score(logp, target):
    h, w = logp.shape
    x, y = np.floor(target).astype(int)
    assert 0 <= x < w and 0 <= y < h
    density = np.exp(logp)
    sd = density.std()
    return {'ll_gain_uniform_bits': float((logp[y, x] + np.log(w*h)) / np.log(2)),
            'nss': float((density[y, x] - density.mean()) / sd) if sd > 1e-15 else 0.0}

def valid_points(path, h, w):
    return (np.isfinite(path).all(axis=1) & (path[:, 0] >= 0) &
            (path[:, 0] < w) & (path[:, 1] >= 0) & (path[:, 1] < h))

def prepare_targets():
    targets, selection = [], []
    selector = np.random.default_rng(SEED + 1)
    for name in selected_names:
        h, w = images[name].shape[:2]
        eligible = []
        invalid_windows = 0
        for subject, path in records[name]:
            good = valid_points(path, h, w)
            indices = []
            for t in range(4, len(path)):
                if good[t-4:t+1].all():
                    indices.append(t)
                else:
                    invalid_windows += 1
            if indices:
                eligible.append((subject, path, indices))
        chosen = selector.permutation(len(eligible))[:SUBJECTS_PER_IMAGE]
        count = 0
        for k in chosen:
            subject, path, indices = eligible[k]
            if MAX_TARGETS_PER_SUBJECT:
                indices = indices[:MAX_TARGETS_PER_SUBJECT]
            for t in indices:
                targets.append(dict(image=name, subject=int(subject), t=int(t),
                                    history=path[t-4:t].copy(), target=path[t].copy()))
                count += 1
        selection.append(dict(image=name, eligible_subjects=len(eligible),
                              selected_subjects=len(chosen), targets=count,
                              invalid_windows=invalid_windows))
    pd.DataFrame(selection).to_csv(OUT / 'selection.csv', index=False)
    if not targets:
        raise RuntimeError('Нет пригодных окон из 4 предыдущих фиксаций и следующей точки.')
    return targets

targets = prepare_targets()
pd.DataFrame([dict(image=t['image'], subject=t['subject'], fixation_index=t['t'],
                   x=t['target'][0], y=t['target'][1], history=t['history'].tolist())
              for t in targets]).to_csv(OUT / 'targets.csv', index=False)
print('Число оцениваемых фиксаций:', len(targets))
display(pd.read_csv(OUT / 'selection.csv'))


## 5. Готовые модели и проверка одного прогноза
Обеим моделям передаётся один и тот же предварительно оценённый center bias из репозитория авторов.
На OSIE он не подгоняется. Изображение подаётся как RGB 0–255: внутренняя нормализация находится в модели.
Модели загружаются по очереди, чтобы экономить память GPU.


In [ ]:
cb_path = download('https://github.com/matthias-k/DeepGaze/releases/download/v1.0.0/centerbias_mit1003.npy',
                   CACHE / 'centerbias_mit1003.npy')
cb_template = np.load(cb_path)

def input_tensors(image):
    h, w = image.shape[:2]
    cb = zoom(cb_template, (h/cb_template.shape[0], w/cb_template.shape[1]), order=0, mode='nearest')
    cb = normalize_log_density(cb)
    image_t = torch.as_tensor(image.transpose(2, 0, 1).copy(), dtype=torch.float32, device=DEVICE)[None]
    cb_t = torch.as_tensor(cb, dtype=torch.float32, device=DEVICE)[None]
    return image_t, cb_t

def predict(model, image, history=None):
    image_t, cb_t = input_tensors(image)
    with torch.inference_mode():
        if history is None:
            result = model(image_t, cb_t)
        else:
            indices = np.asarray(model.included_fixations, dtype=int)
            assert len(history) >= abs(indices.min())
            recent = np.asarray(history)[indices]
            hx = torch.as_tensor(recent[:, 0].copy(), dtype=torch.float32, device=DEVICE)[None]
            hy = torch.as_tensor(recent[:, 1].copy(), dtype=torch.float32, device=DEVICE)[None]
            result = model(image_t, cb_t, hx, hy)
    logp = normalize_log_density(result.detach().cpu().numpy())
    assert logp.shape == image.shape[:2], (logp.shape, image.shape)
    assert abs(np.exp(logp).sum() - 1) < 1e-5
    return logp

import gc
model = deepgaze_pytorch.DeepGazeIIE(pretrained=True).to(DEVICE)
model.eval()
first = targets[0]
smoke = predict(model, images[first['image']])
print('DeepGaze IIE: проверка формы и нормировки пройдена.', smoke.shape)


## 6. Статическая модель и IOR
Карта IIE вычисляется один раз на изображение. Результаты сохраняются по мере обработки.
После обрыва Colab повторный запуск с той же конфигурацией использует сохранённый кэш,
если сама среда и её файлы ещё доступны. При удалении среды кэш теряется.


In [ ]:
maps_dir = OUT / 'maps'
maps_dir.mkdir(exist_ok=True)
rows = []
for name in tqdm(selected_names, desc='DeepGaze IIE'):
    image_targets = [t for t in targets if t['image'] == name]
    if not image_targets:
        continue
    cache_path = maps_dir / (name + '.npy')
    if cache_path.exists():
        logp = normalize_log_density(np.load(cache_path))
    else:
        logp = predict(model, images[name])
        np.save(cache_path, logp)
    for t in image_targets:
        for label, values in [('DeepGaze IIE', logp), ('DeepGaze IIE + IOR', with_ior(logp, t['history']))]:
            rows.append(dict(image=name, subject=t['subject'], t=t['t'], model=label,
                             **score(values, t['target'])))
    pd.DataFrame(rows).to_csv(OUT / 'static_scores.csv', index=False)
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Статические методы завершены.')


## 7. Последовательная модель
Каждое окно истории оценивается отдельно. Это самая длительная часть.
Изображения с примерами выбираются заранее, а не по удачности результата.


In [ ]:
model = deepgaze_pytorch.DeepGazeIII(pretrained=True).to(DEVICE)
model.eval()
smoke = predict(model, images[first['image']], first['history'])
print('DeepGaze III: проверка формы и нормировки пройдена.')

seq_csv = OUT / 'sequential_scores.csv'
seq_rows = pd.read_csv(seq_csv).to_dict('records') if seq_csv.exists() else []
completed = {(r['image'], int(r['subject']), int(r['t'])) for r in seq_rows}
example_targets = []
for name in selected_names:
    candidates = [t for t in targets if t['image'] == name]
    if candidates and len(example_targets) < N_EXAMPLES:
        example_targets.append(candidates[0])
example_keys = {(t['image'], t['subject'], t['t']) for t in example_targets}

for t in tqdm(targets, desc='DeepGaze III'):
    key = (t['image'], t['subject'], t['t'])
    example_path = maps_dir / (t['image'] + '_III.npy')
    if key in completed and (key not in example_keys or example_path.exists()):
        continue
    logp = predict(model, images[t['image']], t['history'])
    if key not in completed:
        seq_rows.append(dict(image=t['image'], subject=t['subject'], t=t['t'],
                             model='DeepGaze III', **score(logp, t['target'])))
        completed.add(key)
        pd.DataFrame(seq_rows).to_csv(seq_csv, index=False)
    if key in example_keys:
        np.save(example_path, logp)

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Все прогнозы готовы.')


## 8. Результаты и неопределённость
Сначала усредняем точки внутри наблюдателя, затем наблюдателей внутри изображения,
затем изображения. Так одно изображение с длинными траекториями не получает больший вес.

95%-е интервалы получены бутстрэпом **по изображениям** (повторными выборками изображений с возвращением).
Разница методов оценивается попарно на тех же изображениях. Интервалы описывают вариативность
изображений данной выборки и не учитывают отдельно вариативность выборки участников.
На малой пробной выборке интервалы нестабильны. Не выбирай параметры IOR и seed по лучшему результату.


In [ ]:
scores = pd.concat([pd.read_csv(OUT / 'static_scores.csv'), pd.read_csv(seq_csv)], ignore_index=True)
metrics = ['ll_gain_uniform_bits', 'nss']
assert not scores.duplicated(['image', 'subject', 't', 'model']).any()
assert scores.groupby(['image', 'subject', 't']).size().eq(3).all()
assert len(scores) == 3 * len(targets)
scores.to_csv(OUT / 'all_fixation_scores.csv', index=False)
per_subject = scores.groupby(['image', 'subject', 'model'])[metrics].mean().reset_index()
per_image = per_subject.groupby(['image', 'model'])[metrics].mean().reset_index()
per_image.to_csv(OUT / 'per_image_scores.csv', index=False)

def bootstrap_mean(values, seed=SEED):
    values = np.asarray(values, dtype=float)
    mean = float(values.mean())
    if len(values) < 2:
        return mean, float('nan'), float('nan')
    generator = np.random.default_rng(seed)
    means = np.array([generator.choice(values, len(values), replace=True).mean()
                      for _ in range(N_BOOTSTRAP)])
    low, high = np.quantile(means, [0.025, 0.975])
    return mean, float(low), float(high)

summary = []
for label in ['DeepGaze IIE', 'DeepGaze IIE + IOR', 'DeepGaze III']:
    for metric in metrics:
        values = per_image.loc[per_image.model == label, metric].values
        mean, low, high = bootstrap_mean(values)
        summary.append(dict(model=label, metric=metric, mean=mean, ci_low=low, ci_high=high,
                            n_images=len(values)))
summary = pd.DataFrame(summary)
summary.to_csv(OUT / 'summary.csv', index=False)

differences = []
for metric in metrics:
    paired = per_image.pivot(index='image', columns='model', values=metric)
    for baseline in ['DeepGaze IIE', 'DeepGaze IIE + IOR']:
        mean, low, high = bootstrap_mean(paired['DeepGaze III'] - paired[baseline])
        differences.append(dict(comparison='DeepGaze III minus ' + baseline, metric=metric,
                                mean_difference=mean, ci_low=low, ci_high=high,
                                n_images=len(paired)))
differences = pd.DataFrame(differences)
differences.to_csv(OUT / 'paired_differences.csv', index=False)
display(summary.round(4))
print('Положительная разница означает преимущество DeepGaze III:')
display(differences.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, metric in zip(axes, metrics):
    values = summary[summary.metric == metric]
    x = np.arange(len(values))
    ax.scatter(x, values['mean'], s=55)
    for i, row in enumerate(values.itertuples()):
        ax.plot([i, i], [row.ci_low, row.ci_high], color='black')
    ax.set_xticks(x, ['IIE', 'IIE + IOR', 'III'])
    ax.set_title(metric + ' (higher is better)')
    ax.grid(axis='y', alpha=.25)
fig.savefig(OUT / 'comparison.png', dpi=200)
plt.show()


## 9. Примеры прогнозов
Показаны четыре предыдущие реальные фиксации (голубая линия, пронумерованная от старой к новой)
и следующая реальная фиксация (зелёная звезда). Цветная карта — вероятность следующей точки.
Это визуализация условного прогноза, а не новая человеческая траектория и не автономная генерация.


In [ ]:
for t in example_targets:
    name = t['image']
    static = np.load(maps_dir / (name + '.npy'))
    predictions = [static, with_ior(static, t['history']), np.load(maps_dir / (name + '_III.npy'))]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    densities = [np.exp(p) for p in predictions]
    common_max = max(p.max() for p in densities)
    for ax, label, p in zip(axes, ['DeepGaze IIE', 'DeepGaze IIE + IOR', 'DeepGaze III'], densities):
        ax.imshow(images[name])
        overlay = ax.imshow(p, cmap='magma', alpha=.6, vmin=0, vmax=common_max)
        ax.plot(t['history'][:, 0], t['history'][:, 1], 'o-', color='cyan', lw=2)
        for i, (x, y) in enumerate(t['history']):
            ax.annotate(str(i+1), (x, y), color='white', fontsize=10)
        ax.scatter(*t['target'], marker='*', s=220, c='lime', edgecolors='black')
        ax.set_title(label)
        ax.axis('off')
    fig.suptitle(f"OSIE {name}, subject {t['subject']}, next fixation {t['t']+1}")
    fig.colorbar(overlay, ax=axes, shrink=.7, label='Probability per pixel')
    fig.savefig(OUT / (name + '_predictions.png'), dpi=180)
    plt.show()


## 10. Сохранить результаты
Архив содержит таблицы, графики, настройки, выбранные точки, версии библиотек и описание протокола.
Веса и исходные изображения в архив не включаются. Блокнот с выполненными ячейками сохрани отдельно
через **Файл → Скачать → Скачать IPYNB**.

### Как интерпретировать
- Превосходство III по этим метрикам означает более точный прогноз следующей реальной точки
  при предоставленной истории **в данных условиях**.
- Сравнение разных моделей не изолирует причинный эффект истории. IOR тоже использует историю,
  но посредством заданного правила.
- Это не проверка длительности фиксаций, остановки просмотра или полного сгенерированного scanpath.
- MIT1003 использовался для обучения выпущенных DeepGaze; он намеренно не выбран тестовым набором.
  OSIE — другой набор, но перед публикацией нужно проверить возможные пересечения с данными
  предобучения всех компонентов и описать различия условий предъявления изображений.
- Модели используются без адаптации масштаба к угловому размеру стимула OSIE; это ограничение
  переноса между наборами данных и возможный источник различий результатов.
- Числовые результаты появляются только после выполнения блокнота. Не подставляй выдуманные значения.

### Источники для статьи и воспроизводимости
- [DeepGaze: код и обученные модели](https://github.com/matthias-k/DeepGaze)
- [DeepGaze III, Kümmerer et al., 2022](https://doi.org/10.1167/jov.22.5.7)
- [DeepGaze IIE, Linardos et al., 2021](https://arxiv.org/abs/2105.12441)
- [OSIE: Predicting Human Gaze Beyond Pixels](https://github.com/NUS-VIP/predicting-human-gaze-beyond-pixels)
- [Загрузчик OSIE в pysaliency](https://github.com/matthias-k/pysaliency/blob/master/pysaliency/external_datasets/osie.py)
- [Условия бесплатного Colab](https://research.google.com/colaboratory/faq.html)

### Если возникла ошибка
- **Нет GPU / нехватка памяти:** включи GPU, перезапусти среду и выполняй ячейки заново.
- **Ошибка скачивания:** повтори ячейку; доступность GitHub и серверов весов зависит от сети Colab.
- **Ошибка импорта:** перезапусти среду после установки. Сохрани полный текст ошибки, если она повторяется.
- **Среда удалилась:** загрузи блокнот заново; файлы в `/content` временные.


In [ ]:
import platform, shutil
manifest = dict(config, device=DEVICE, python=platform.python_version(),
                torch=torch.__version__, numpy=np.__version__,
                centerbias_sha256=hashlib.sha256(cb_path.read_bytes()).hexdigest(),
                n_fixations=len(targets), completed_utc=pd.Timestamp.now(tz='UTC').isoformat())
(OUT / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
(OUT / 'requirements_actual.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True))
protocol = f"""Сравнение прогнозирования следующей фиксации на OSIE.
Изображений запрошено: {N_IMAGES}; оценено: {per_image.image.nunique()}.
Оценено реальных фиксаций: {len(targets)}. Seed: {SEED}.
Окно: четыре предыдущие реальные фиксации, в порядке model.included_fixations.
Предобученные DeepGaze IIE, IIE + гауссово подавление IOR и DeepGaze III.
IOR strength: {IOR_STRENGTH}; sigma/min(H,W): {IOR_SIGMA_FRACTION}.
Исходное разрешение OSIE. Center bias из обучающего MIT1003, без подгонки на OSIE.
Первичная метрика: log2(p(target)*W*H), выигрыш относительно равномерного распределения.
Дополнительная: NSS. Пиксель цели: floor(x), floor(y).
Средние: фиксации -> наблюдатель -> изображение -> набор.
Парный bootstrap по изображениям, {N_BOOTSTRAP} повторов, percentile 95% CI.
Разные архитектуры не позволяют изолировать эффект памяти.
Не оценивались свободная генерация scanpath, duration и остановка.
Версия кода: {DEEPGAZE_SHA}.
Пробная малая выборка не даёт основания для широких выводов.
Перед публикацией проверить пересечения предобучающих данных и условия предъявления OSIE.
"""
(OUT / 'protocol_ru.txt').write_text(protocol, encoding='utf-8')
export = BASE / ('export_' + run_id)
export.mkdir(exist_ok=True)
for path in OUT.iterdir():
    if path.is_file():
        shutil.copy2(path, export / path.name)
archive = shutil.make_archive(str(BASE / ('attention_results_' + run_id)), 'zip', export)
print('Готово:', archive)
from google.colab import files
files.download(archive)
